# Milestone 1: EDA and Baseline Setup

In [2]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/train.csv')


## Question 1
Calculate the frequency distribution of the correct answer (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [3]:
ans_counts = df['answer'].value_counts()
print("Frequency distribution:")
print(ans_counts)

sum_most_least = ans_counts.max() + ans_counts.min()
print('\nSum of most and least frequent:', sum_most_least)


Frequency distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Sum of most and least frequent: 814


## Question 2
After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?

In [4]:
def clean_text(text):
    text = str(text).lower()
    for p in string.punctuation:
        text = text.replace(p, '')
    return text

cleaned_prompts = df['prompt'].apply(clean_text)
words = set()
for prompt in cleaned_prompts:
    words.update(prompt.split())
print('Total unique words in cleaned prompt column:', len(words))


Total unique words in cleaned prompt column: 859


## Question 3
Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [5]:
# Row ID 1 is the first row (index 0)
prompt_1 = cleaned_prompts.iloc[0]
filtered_words = [word for word in prompt_1.split() if word not in ENGLISH_STOP_WORDS]
print('Words left in Row ID 1:', len(filtered_words))


Words left in Row ID 1: 13


## Question 4
Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [6]:
all_text = df['prompt'].tolist() + df['A'].tolist() + df['B'].tolist() + df['C'].tolist() + df['D'].tolist() + df['E'].tolist()
# Convert everything to string to handle any NaNs
all_text = [str(t) for t in all_text]

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(all_text)
print('Total number of feature columns (vocabulary size):', len(vectorizer.get_feature_names_out()))


Total number of feature columns (vocabulary size): 2762


## Question 5
Using the TF-IDF vectorizer fitted in Question 4, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [7]:
# Row ID 1 (index 0)
p1 = str(df.iloc[0]['prompt'])
a1 = str(df.iloc[0]['A'])

# Transform using the fitted vectorizer
vectors = vectorizer.transform([p1, a1])
sim = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
print(f'Cosine similarity (Prompt 1 vs Option A): {sim:.4f}')


Cosine similarity (Prompt 1 vs Option A): 0.2328


## Question 6
Expand the logic from Question 5: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options. Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [8]:
matches = 0
options_cols = ['A', 'B', 'C', 'D', 'E']

for idx, row in df.iterrows():
    p_text = str(row['prompt'])
    opts_text = [str(row[opt]) for opt in options_cols]
    
    vecs = vectorizer.transform([p_text] + opts_text)
    # vecs[0] is prompt, vecs[1:6] are options
    sims = cosine_similarity(vecs[0:1], vecs[1:]).flatten()
    
    best_opt_idx = np.argmax(sims)
    best_opt_letter = options_cols[best_opt_idx]
    
    if best_opt_letter == row['answer']:
        matches += 1

print(f'Percentage of correct matches: {(matches / len(df)) * 100:.2f}%')


Percentage of correct matches: 13.70%


## Question 7
If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

In [9]:
# MAP@3 definition: if the correct answer is at rank k (1, 2, or 3), the score is 1/k.
# Here, C is the 1st prediction (rank 1).
# Score = 1/1 = 1.0
print('MAP@3 score for Truth=C, Pred=C A B:', 1.0)


MAP@3 score for Truth=C, Pred=C A B: 1.0


## Question 8
If the ground truth answer for a question is B, what is the MAP@3 score if a model predicts D B E?

In [10]:
# Here, B is the 2nd prediction (rank 2).
# Score = 1/2 = 0.5
print('MAP@3 score for Truth=B, Pred=D B E:', 0.5)


MAP@3 score for Truth=B, Pred=D B E: 0.5


## Question 9
The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this 'Majority Class' baseline on train.csv?

In [11]:
# Find top 3 most frequent answers
top3_frequent = ans_counts.index[:3].tolist()
print('Top 3 frequent answers:', top3_frequent)

score = 0.0
for true_ans in df['answer']:
    if true_ans == top3_frequent[0]:
        score += 1.0
    elif true_ans == top3_frequent[1]:
        score += 1.0 / 2.0
    elif true_ans == top3_frequent[2]:
        score += 1.0 / 3.0

map3_majority = score / len(df)
print('MAP@3 of Majority Class baseline:', map3_majority)


Top 3 frequent answers: ['B', 'C', 'A']
MAP@3 of Majority Class baseline: 0.4212500000000017


## Question 10
The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [12]:
total_score = 0.0

for idx, row in df.iterrows():
    p_text = str(row['prompt'])
    opts_text = [str(row[opt]) for opt in options_cols]
    
    vecs = vectorizer.transform([p_text] + opts_text)
    sims = cosine_similarity(vecs[0:1], vecs[1:]).flatten()
    
    # Sort indices descending
    sorted_idx = np.argsort(sims)[::-1]
    
    # Get top 3 predicted letters
    top3_preds = [options_cols[i] for i in sorted_idx[:3]]
    
    true_ans = row['answer']
    
    if true_ans in top3_preds:
        rank = top3_preds.index(true_ans) + 1
        total_score += 1.0 / rank

map3_tfidf = total_score / len(df)
print('MAP@3 of TF-IDF baseline:', map3_tfidf)


MAP@3 of TF-IDF baseline: 0.2761666666666658
